In [ ]:
import os
import torch
import torchaudio
import matplotlib.pyplot as plt
import pandas as pd
import pyroomacoustics as pra
import librosa

___
# Analyze Database

In [ ]:
ROOT_TRN = '/home/ovistetom/Documents/data/MIX-EARS-WHAM/reference/tst'
SAMPLE_RATE = 16000
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
info = {}

for i, name_sample in enumerate(os.listdir(ROOT_TRN)):
    path_sample = os.path.join(ROOT_TRN, name_sample)

    noisy_mixture = torchaudio.load(os.path.join(path_sample, 'noisy_mixture.flac'), normalize=True, channels_first=True)[0]
    overall_noise = torchaudio.load(os.path.join(path_sample, 'overall_noise.flac'), normalize=True, channels_first=True)[0]
    ambient_noise = torchaudio.load(os.path.join(path_sample, 'ambient_noise.flac'), normalize=True, channels_first=True)[0]
    target_speech = torchaudio.load(os.path.join(path_sample, 'target_speech.flac'), normalize=True, channels_first=True)[0]
    reverb_speech = torchaudio.load(os.path.join(path_sample, 'reverb_speech.flac'), normalize=True, channels_first=True)[0]
    interf_speech = torchaudio.load(os.path.join(path_sample, 'interf_speech.flac'), normalize=True, channels_first=True)[0]

    target_speech = target_speech + reverb_speech

    length = noisy_mixture.size(1)
    sir = target_speech.pow(2.0).sum() / interf_speech.pow(2.0).sum()
    sir = 10*torch.log10(sir).item()
    snr = target_speech.pow(2.0).sum() / ambient_noise.pow(2.0).sum()
    snr = 10*torch.log10(snr).item()
    sor = target_speech.pow(2.0).sum() / overall_noise.pow(2.0).sum()
    sor = 10*torch.log10(sor).item()

    info[i] = [name_sample, length/16000, sir, snr, sor]

In [ ]:
# Dataframe from dict
df = pd.DataFrame.from_dict(info, orient='index', columns=['NAME', 'LEN', 'SIR', 'SNR', 'SOR'])

In [ ]:
df.head()

In [ ]:
fig, axs = plt.subplots(1,1)
axs.hist(df['LEN']);
axs.set_title('Duration [s]')
fig.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1,1)
axs.hist(df['SIR']);
axs.set_xlabel('SIR [dB]')
axs.set_ylabel('Count')
axs.set_title('Signal-to-Interference Ratio')
fig.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1,1)
axs.hist(df['SNR']);
axs.set_xlabel('SNR [dB]')
axs.set_ylabel('Count')
axs.set_title('Signal-to-Noise Ratio')
fig.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1,1)
axs.hist(df['SNR']);
axs.set_xlabel('SNR [dB]')
axs.set_ylabel('Count')
axs.set_title('Signal-to-Noise Ratio (Overall Noise)')
fig.tight_layout()
plt.show()